In [ ]:
import sqlite3
from DBTypes import *
from SwingDBTypes import *

In [ ]:
db = sqlite3.connect("../../Db/BaseballStats.db")
swingDb = sqlite3.connect("../../SwingDecisionsDb/SwingDecisions.db")

In [ ]:
from enum import Enum


class PHYA_Y_Column(Enum):
    AVG = "AVG"
    OBP = "OBP"
    SLG = "SLG"
    ISO = "ISO"
    wOBA = "wOBA"
    wRC = "wRC"
    HR = "HR"
    BBPerc = "BBPerc"
    KPerc = "KPerc"


class SRA_X_Option(Enum):
    ValuePer100Swings = "ValuePer100Swings"
    ValuePer100NonSwings = "ValuePer100NonSwings"
    ValuePer100Events = "ValuePer100Events"

In [ ]:
# X option -> (value_func, count_func), keyed by enum member.
SRA_X_OPTIONS = {
    SRA_X_Option.ValuePer100Swings: {
        "value_func": lambda s: s.ValuePer100Swings,
        "count_func": lambda s: s.NumSwings,
    },
    SRA_X_Option.ValuePer100NonSwings: {
        "value_func": lambda s: s.ValuePer100NonSwings,
        "count_func": lambda s: s.NumNonSwings,
    },
    SRA_X_Option.ValuePer100Events: {
        "value_func": lambda s: (
            (s.ValueSwings + s.ValueNonSwings)
            / (s.NumSwings + s.NumNonSwings) * 100.0
        ) if (s.NumSwings + s.NumNonSwings) > 0 else 0.0,
        "count_func": lambda s: s.NumSwings + s.NumNonSwings,
    },
}

In [ ]:
def Get_Swing_Rows(swingCursor: 'sqlite3.Cursor',
                   levelId: int = 1) -> list['DB_SwingResultAggregation']:
    conditional = ("WHERE Month IS NULL "
                   "AND PitcherId IS NULL "
                   "AND PitchGroup = 1 "
                   "AND LevelId = ?")
    values = (levelId,)
    return DB_SwingResultAggregation.Select_From_DB(swingCursor, conditional, values)

In [ ]:
def Get_Weighted_PHYA_Value(hitterCursor: 'sqlite3.Cursor',
                            mlbId: int,
                            year: int,
                            yColumn: PHYA_Y_Column,
                            paRequirement: int,
                            levelId: int = 1) -> float | None:
    conditional = "WHERE mlbId = ? AND year = ? AND levelId = ?"
    values = (mlbId, year, levelId)
    rows = DB_Player_Hitter_YearAdvanced.Select_From_DB(hitterCursor, conditional, values)

    totalPA = sum(r.PA for r in rows)
    if totalPA < paRequirement or totalPA == 0:
        return None

    weighted = sum(getattr(r, yColumn.value) * r.PA for r in rows) / totalPA
    return weighted, totalPA

In [ ]:
def Build_Regression_Dataset(hitterCursor: 'sqlite3.Cursor',
                             swingCursor: 'sqlite3.Cursor',
                             xVariable: SRA_X_Option,
                             yColumn: PHYA_Y_Column,
                             numSwingsRequirement: int,
                             paRequirement: int,
                             levelId: int = 1
                             ) -> tuple[list[float], list[float], list[int]]:
    value_func = SRA_X_OPTIONS[xVariable]["value_func"]
    count_func = SRA_X_OPTIONS[xVariable]["count_func"]

    swingRows = Get_Swing_Rows(swingCursor, levelId)

    xs: list[float] = []
    ys: list[float] = []
    pas: list[int] = []

    for s in swingRows:
        if count_func(s) < numSwingsRequirement:
            continue
        result = Get_Weighted_PHYA_Value(hitterCursor, s.HitterId, s.Year,
                                         yColumn, paRequirement, levelId)
        if result is None:
            continue
        y, totalPA = result
        xs.append(value_func(s))
        ys.append(y)
        pas.append(totalPA)

    return xs, ys, pas

In [ ]:
import numpy as np

def Compute_Rolling_Average(x: np.ndarray,
                            y: np.ndarray,
                            pa: np.ndarray,
                            halfWidth: float,
                            numPoints: int = 200
                            ) -> tuple[np.ndarray, np.ndarray]:
    """PA-weighted rolling average of y over an X window of +-halfWidth.

    Returns (grid_x, avg_y) with NaNs where a window contains no points.
    """
    grid_x = np.linspace(x.min(), x.max(), numPoints)
    avg_y = np.full(numPoints, np.nan)

    for i, cx in enumerate(grid_x):
        mask = (x >= cx - halfWidth) & (x <= cx + halfWidth)
        w = pa[mask]
        if w.sum() > 0:
            avg_y[i] = np.average(y[mask], weights=w)

    return grid_x, avg_y

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def Plot_Bubble_Regression(xs: list[float],
                           ys: list[float],
                           pas: list[int],
                           xLabel: str,
                           yLabel: str,
                           numSwingsRequirement: int,
                           paRequirement: int,
                           rollingHalfWidth: float = 0.05,
                           areaPerPA: float = 0.5) -> None:
    if len(xs) < 2:
        raise ValueError("Not enough data points to run a regression.")

    x = np.array(xs, dtype=float)
    y = np.array(ys, dtype=float)
    pa = np.array(pas, dtype=float)

    sizes = pa * areaPerPA

    # PA-weighted least squares.
    m, b = np.polyfit(x, y, 1, w=pa)
    y_pred = m * x + b

    y_bar = np.average(y, weights=pa)
    ss_res = np.sum(pa * (y - y_pred) ** 2)
    ss_tot = np.sum(pa * (y - y_bar) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else float("nan")

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(x, y, s=sizes,
               facecolor="#9EC3E6", edgecolors="#2C3E50", linewidths=0.5,
               alpha=0.55, zorder=2, label="Hitters (area ∝ PA)")

    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b,
            color="#D1495B", linewidth=2.2, zorder=3,
            label=f"PA-weighted: y = {m:.3f}x + {b:.3f}\n$R^2$ = {r2:.3f}  (n = {len(x)})")

    # PA-weighted rolling average to reveal non-linearities.
    grid_x, avg_y = Compute_Rolling_Average(x, y, pa, rollingHalfWidth)
    ax.plot(grid_x, avg_y,
            color="#1B1B3A", linewidth=2.4, linestyle="--", zorder=4,
            label=f"Rolling avg (±{rollingHalfWidth:g})")

    ax.set_xlabel(xLabel)
    ax.set_ylabel(f"{yLabel} (PA-weighted)")
    ax.set_title(f"{xLabel} vs {yLabel} "
                 f"(count ≥ {numSwingsRequirement}, PA ≥ {paRequirement})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
def Run_Bubble_Regression(db: 'sqlite3.Connection',
                          swingDb: 'sqlite3.Connection',
                          xVariable: SRA_X_Option,
                          yColumn: PHYA_Y_Column,
                          numSwingsRequirement: int,
                          paRequirement: int,
                          levelId: int = 1,
                          rollingHalfWidth: float = 0.05) -> None:
    hitterCursor = db.cursor()
    swingCursor = swingDb.cursor()

    xs, ys, pas = Build_Regression_Dataset(
        hitterCursor, swingCursor,
        xVariable, yColumn,
        numSwingsRequirement, paRequirement,
        levelId
    )

    print(f"Matched {len(xs)} hitters meeting the constraints.")
    Plot_Bubble_Regression(xs, ys, pas, xVariable.value, yColumn.value,
                           numSwingsRequirement, paRequirement,
                           rollingHalfWidth=rollingHalfWidth)

In [ ]:
Run_Bubble_Regression(
    db, swingDb,
    xVariable=SRA_X_Option.ValuePer100Events,
    yColumn=PHYA_Y_Column.wRC,
    numSwingsRequirement=100,
    paRequirement=200,
    rollingHalfWidth=0.05,
)